in this section i want and mask with orginal images and extract features from this images


part1: AND images with masks

AND segmented image with image and save

In [2]:
import os, re, glob, cv2, numpy as np
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms
from sklearn.model_selection import KFold
from attention_unet import AttentionUNet

images_dir = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images")
labels_dir = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels")
model_path = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\attention_unet_fold_2.pth")
output_dir = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_fold2")
output_dir.mkdir(parents=True, exist_ok=True)

def get_id(p: Path) -> int:
    return int(re.findall(r'\d+', p.stem)[0])

mask_map = {get_id(p): p for p in labels_dir.glob("*.*")}

pairs = []
for img_path in sorted(images_dir.glob("*.*"), key=get_id):
    fid = get_id(img_path)
    if fid in mask_map:
        pairs.append((img_path, mask_map[fid]))


images_list, masks_list = zip(*pairs)    

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionUNet().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

tfm = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
train_idx, _ = list(kf.split(images_list))[2]     

for idx in train_idx:
    img_path  = Path(images_list[idx])
    mask_path = Path(masks_list[idx])
    file_id   = get_id(img_path)       

    img_pil = Image.open(img_path).convert("RGB")
    img_t   = tfm(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.sigmoid(model(img_t))

    pred_bin = (pred.cpu().squeeze().numpy() > 0.5).astype(np.uint8)
    inv_mask = (1 - pred_bin) * 255   

    img_np  = (img_t.cpu().squeeze().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    and_res = cv2.bitwise_and(img_np, img_np, mask=inv_mask)

    out_path = output_dir / f"{file_id}.png"
    Image.fromarray(and_res).save(out_path)

print(f"Done {output_dir}")


In [1]:
import os, re, glob, cv2, numpy as np
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms
from sklearn.model_selection import KFold
from attention_unet import AttentionUNet


images_dir = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images")
labels_dir = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels")
model_path = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\attention_unet_fold_2.pth")
output_test_dir = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_test")

output_test_dir.mkdir(parents=True, exist_ok=True)

# تابع استخراج شناسه از نام فایل
def get_id(p: Path) -> int:
    return int(re.findall(r'\d+', p.stem)[0])

# لیست‌های داده‌ها
mask_map = {get_id(p): p for p in labels_dir.glob("*.*")}
pairs = []
for img_path in sorted(images_dir.glob("*.*"), key=get_id):
    fid = get_id(img_path)
    if fid in mask_map:
        pairs.append((img_path, mask_map[fid]))
images_list, masks_list = zip(*pairs)    

# بارگذاری مدل
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionUNet().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# تبدیل‌های لازم
tfm = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# تولید اندیس‌های تست واقعی (نه آموزشی!)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
_, test_idx = list(kf.split(images_list))[2]  # انتخاب fold 2 برای تست

# پردازش داده‌های تست واقعی
for idx in test_idx:  # فقط داده‌های تست!
    img_path  = Path(images_list[idx])
    mask_path = Path(masks_list[idx])
    file_id   = get_id(img_path)       

    img_pil = Image.open(img_path).convert("RGB")
    img_t   = tfm(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.sigmoid(model(img_t))

    pred_bin = (pred.cpu().squeeze().numpy() > 0.5).astype(np.uint8)
    inv_mask = (1 - pred_bin) * 255   

    img_np  = (img_t.cpu().squeeze().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    and_res = cv2.bitwise_and(img_np, img_np, mask=inv_mask)

    out_path = output_test_dir / f"{file_id}.png"
    Image.fromarray(and_res).save(out_path)

print(f"Test data processed: {output_test_dir}")